# Pharmacon — Writer-Adaptive Prescription OCR
## Empirical Benchmark: Generic vs Adapted TrOCR on Sealed Lines (T01–T08)

**Authoritative Protocol**: `Pharmacon_Writer_Calibration_Protocol.pdf`  
**Course**: Software Engineering (UCS503), Lab Project  

### Pipeline Structure:
1. **Fixed-Form Cropping**: Perspective warp + per-line box extraction (with pre-cropped sheet fallback).
2. **Strict Data Splits**: 16 Enrollment (`E01–E16`), 4 Validation (`V01–V04`), 8 Sealed Held-Out Test (`T01–T08`).
3. **Generic Baseline**: Inference with pretrained `trocr-small-handwritten` (decoder frozen).
4. **Writer Adaptation**: LoRA rank 8 on encoder layers 10 & 11 (24,576 parameters, 0.0399%).
5. **Early Stopping**: Monitored on validation loss (`V01–V04`) to prevent overfitting.
6. **Empirical Benchmark**: Side-by-side CER/WER comparison on sealed test lines (`T01–T08`).

In [ ]:
# Cell 1: Install dependencies
!pip install -q "sentencepiece>=0.2.0" "protobuf>=3.20.3" "transformers>=4.40.0" "peft>=0.10.0" "accelerate>=0.28.0" "pymupdf>=1.24.0" "rapidfuzz>=3.0.0"


In [ ]:
# Cell 2: Self-healing restart check
import os
try:
    import sentencepiece
    print(f"[OK] sentencepiece {sentencepiece.__version__} ready.")
except ImportError:
    print("Restarting kernel to register sentencepiece...")
    os.kill(os.getpid(), 9)

In [ ]:
# Cell 3: Environment Configuration
import os, sys

# Set ENV to 'colab' when running on Google Colab, or 'local' on your local workstation
ENV = 'colab' if os.path.exists('/content') else 'local'

if ENV == 'colab':
    REPO_ROOT  = '/content'
    IMAGES_DIR = '/content'
    PDF_PATH   = '/content/Pharmacon_Writer_Calibration_Protocol.pdf'
else:
    REPO_ROOT  = os.path.abspath('..')
    IMAGES_DIR = os.path.join(REPO_ROOT, 'sheet_photos')
    PDF_PATH   = os.path.join(REPO_ROOT, 'Pharmacon_Writer_Calibration_Protocol.pdf')

sys.path.insert(0, REPO_ROOT)
print(f"Environment : {ENV}")
print(f"PDF exists  : {os.path.exists(PDF_PATH)}")

In [ ]:
# Cell 4: Imports and Device Setup
import copy, cv2, torch
import numpy as np
from transformers import AutoImageProcessor, XLMRobertaTokenizer, TrOCRProcessor, VisionEncoderDecoderModel
from peft import LoraConfig, get_peft_model
import pharmacon_ai.crop as crop_mod
from pharmacon_ai import process_photo, evaluate_condition, print_comparison

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute device: {device}",
      f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else '(CPU)')

In [ ]:
# Cell 5: Map Sheets to PDF Pages and Verify Files
WRITER_ID = "W02"  # Set to "W01" or "W02"

if WRITER_ID == "W01":
    PAGE_PHOTOS = {
        2: os.path.join(IMAGES_DIR, "TrainS1.jpg"),      # E01-E04
        3: os.path.join(IMAGES_DIR, "TrainS2.jpg"),      # E05-E08
        4: os.path.join(IMAGES_DIR, "TrainS3.jpg"),      # E09-E12
        5: os.path.join(IMAGES_DIR, "TrainsS4.jpg"),     # E13-E16
        6: os.path.join(IMAGES_DIR, "ValidationS.jpg"), # V01-V04, R01-R04
        7: os.path.join(IMAGES_DIR, "HeldOutS1.jpg"),   # T01-T04
        8: os.path.join(IMAGES_DIR, "HeldOutS2.jpg"),   # T05-T08
    }
elif WRITER_ID == "W02":
    PAGE_PHOTOS = {
        2: os.path.join(IMAGES_DIR, "W2TS1.jpg"),       # E01-E04
        3: os.path.join(IMAGES_DIR, "W2TS2.jpg"),       # E05-E08
        4: os.path.join(IMAGES_DIR, "W2TS3.jpg"),       # E09-E12
        5: os.path.join(IMAGES_DIR, "W2TS4.jpg"),       # E13-E16
        6: os.path.join(IMAGES_DIR, "W2VS.jpg"),        # V01-V04, R01-R04
        7: os.path.join(IMAGES_DIR, "W2HS1.jpg"),       # T01-T04
        8: os.path.join(IMAGES_DIR, "W2HS2.jpg"),       # T05-T08
    }

print(f"Checking sheet photographs for Writer {WRITER_ID}:")
for p_idx, p_path in PAGE_PHOTOS.items():
    status = "[OK]" if os.path.exists(p_path) else "[MISSING]"
    print(f"  Page {p_idx}: {status} {os.path.basename(p_path)}")


In [ ]:
# Cell 6: Extract Line Crops with Pre-Cropped Sheet Fallback
all_lines = {}
for p_idx, p_path in PAGE_PHOTOS.items():
    if not os.path.exists(p_path):
        print(f"  [SKIP] Missing page {p_idx}: {p_path}")
        continue
    lines = process_photo(p_path, PDF_PATH, p_idx, writer_id=WRITER_ID)
    all_lines.update(lines)
    print(f"  Page {p_idx} cropped: {list(lines.keys())}")

print(f"\nTotal lines loaded: {len(all_lines)} / 28")

In [ ]:
# Cell 7: Partition Splits and Seal Check
train_ids = [f"E{i:02d}" for i in range(1, 17)]
val_ids   = [f"V{i:02d}" for i in range(1, 5)]
test_ids  = [f"T{i:02d}" for i in range(1, 9)]

train_crops  = [cv2.cvtColor(all_lines[lid]['image'], cv2.COLOR_BGR2RGB) for lid in train_ids]
train_labels = [all_lines[lid]['label'] for lid in train_ids]

val_crops    = [cv2.cvtColor(all_lines[lid]['image'], cv2.COLOR_BGR2RGB) for lid in val_ids]
val_labels   = [all_lines[lid]['label'] for lid in val_ids]

test_crops   = {lid: cv2.cvtColor(all_lines[lid]['image'], cv2.COLOR_BGR2RGB) for lid in test_ids}
test_labels  = {lid: all_lines[lid]['label'] for lid in test_ids}

# Strict seal verification
overlap = set(train_ids).intersection(test_ids)
assert not overlap, f"SEAL VIOLATION: Overlap detected: {overlap}"
del all_lines  # Delete raw source to guarantee hold-out isolation

print('[SEALED] Protocol splits isolated successfully:')
print(f"  Enrollment (E01-E16) : {len(train_crops)} lines (Training)")
print(f"  Validation (V01-V04) : {len(val_crops)} lines (Early Stopping)")
print(f"  Held-Out   (T01-T08) : {len(test_crops)} lines (Strictly Sealed)")

In [ ]:
# Cell 8: Load Pretrained TrOCR-small Baseline
MODEL_ID = 'microsoft/trocr-small-handwritten'
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
tokenizer       = XLMRobertaTokenizer.from_pretrained(MODEL_ID)
processor       = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to(device)
model.config.decoder_start_token_id = processor.tokenizer.eos_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

print(f"[OK] Loaded baseline model: {MODEL_ID}")

In [ ]:
# Cell 9: Evaluate Generic Baseline on Sealed T01-T08
print('Evaluating Generic Baseline on sealed lines T01-T08...')
model.eval()
generic_preds = {}

with torch.no_grad():
    for lid, img in test_crops.items():
        pv = processor(images=img, return_tensors='pt').pixel_values.to(device)
        ids = model.generate(
            pv, max_new_tokens=64, num_beams=4, early_stopping=True,
            no_repeat_ngram_size=3, repetition_penalty=1.3
        )
        generic_preds[lid] = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

true_generic_results = evaluate_condition(generic_preds, test_labels)
gm = true_generic_results['macro']
print(f"Generic MACRO: CER = {gm['cer']:.4f} | WER = {gm['wer']:.4f}")

In [ ]:
# Cell 10: Configure LoRA Adapter on Last Two Visual Blocks
try:
    import peft.import_utils; peft.import_utils.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao; peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

targets = [
    'encoder.layers.10.attention.q_proj',
    'encoder.layers.10.attention.v_proj',
    'encoder.layers.11.attention.q_proj',
    'encoder.layers.11.attention.v_proj',
]

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.15, target_modules=targets, bias='none')
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
# Cell 11: Train Adapter on E01-E16 with Early Stopping on V01-V04
opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.01)
best_val_loss, best_weights, patience, p_count = float('inf'), None, 4, 0

print(f"Training LoRA adapter on {len(train_crops)} lines with validation monitoring on {len(val_crops)} lines...\n")

for epoch in range(1, 41):
    peft_model.train()
    t_loss = 0.0
    for img, text in zip(train_crops, train_labels):
        pv = processor(images=img, return_tensors='pt').pixel_values.to(device)
        lbl = processor.tokenizer(text=text, return_tensors='pt').input_ids.to(device)
        loss = peft_model(pixel_values=pv, labels=lbl).loss
        opt.zero_grad(); loss.backward(); opt.step()
        t_loss += loss.item()
    t_loss /= len(train_crops)

    peft_model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for img, text in zip(val_crops, val_labels):
            pv = processor(images=img, return_tensors='pt').pixel_values.to(device)
            lbl = processor.tokenizer(text=text, return_tensors='pt').input_ids.to(device)
            v_loss += peft_model(pixel_values=pv, labels=lbl).loss.item()
    v_loss /= len(val_crops)

    if epoch % 5 == 0 or v_loss < best_val_loss:
        print(f"Epoch {epoch:02d} | Train: {t_loss:.4f} | Val: {v_loss:.4f}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_weights = copy.deepcopy(peft_model.state_dict())
        p_count = 0
    else:
        p_count += 1
        if p_count >= patience:
            print(f"Early stopping triggered at epoch {epoch}. Restoring best checkpoint (Val Loss: {best_val_loss:.4f}).")
            break

if best_weights:
    peft_model.load_state_dict(best_weights)
print('Adapter training completed.')

In [ ]:
# Cell 12: Evaluate Adapted Model on Sealed T01-T08 and Print Report
print('Evaluating Adapted Model on sealed lines T01-T08...')
peft_model.eval()
adapted_preds = {}

with torch.no_grad():
    for lid, img in test_crops.items():
        pv = processor(images=img, return_tensors='pt').pixel_values.to(device)
        ids = peft_model.generate(
            pv, max_new_tokens=64, num_beams=4, early_stopping=True,
            no_repeat_ngram_size=3, repetition_penalty=1.3
        )
        adapted_preds[lid] = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

adapted_results = evaluate_condition(adapted_preds, test_labels)

print('\n' + '='*80)
print(f"PHARMACON — FINAL BENCHMARK (Writer {WRITER_ID}: Generic vs Adapted on Sealed T01-T08)")
print('='*80)
print_comparison(true_generic_results, adapted_results, label_a='Generic', label_b='Adapted')

In [ ]:
# Cell 13: Clinical Resolution via FormularyMatcher
from pharmacon_ai.formulary import FormularyMatcher
matcher = FormularyMatcher()

print("
" + "="*80)
print(f"CLINICAL RESOLUTION (Formulary Matching on Writer {WRITER_ID} Held-Out Lines)")
print("="*80)
for lid in sorted(test_labels.keys()):
    raw = adapted_preds[lid]
    target = test_labels[lid]
    res = matcher.resolve(raw)
    resolved_str = "; ".join(f"{d.matched_drug_name} {d.strength or ''} {d.form or ''}".strip() for d in res)
    print(f"Line {lid}:")
    print(f"  Target   : {target}")
    print(f"  Raw OCR  : {raw}")
    print(f"  Resolved : {resolved_str or '[GATE_REJECTED / UNMATCHED]'}")
